In [96]:
# IMPORT LIBRARY
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import os
import joblib

In [97]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

In [98]:
# ==========================================================
# IMPORT DAMAGE FEATURE ENGINEERING
# ==========================================================

from damage_feature_engineering import (
    clean_damage,
    extract_damage_features
)

In [102]:
# ================
# DATA SELECTION
# ================
print("="*70)
print("MEMBACA DATASET")
print("="*70)
EXCEL_FILE = "kejadian bencana_2020-2025.xlsx"

all_df=[]

for year in range(2020,2026):
    temp=pd.read_excel(
        EXCEL_FILE,
        sheet_name=str(year)
    )

    temp["Year"]=year
    all_df.append(temp)

df=pd.concat(
    all_df,
    ignore_index=True
)

# ==========================================================
# NORMALISASI NAMA KEJADIAN BERDASARKAN JUKLAK BNPB
# ==========================================================

replace_map = {

    # ======================================================
    # BANJIR
    # ======================================================

    "Banjir": "Banjir",
    "Banjir Rob": "Banjir Rob",
    "Banjir Bandang": "Banjir Bandang",
    "Banjir dan Tanah Longsor": "Banjir dan Tanah Longsor",
    "Banjir Drainase & Selokan": "Banjir Drainase & Selokan",
    "Banjir Waduk": "Banjir Waduk",
    "Banjir Genangan": "Banjir Genangan",
    "Tanggul Jebol": "Tanggul Jebol",

    # ======================================================
    # TANAH LONGSOR
    # ======================================================

    "Tanah Longsor": "Tanah Longsor",
    "Gerakan Tanah": "Gerakan Tanah",

    # ======================================================
    # GELOMBANG PASANG DAN ABRASI
    # ======================================================

    "Gelombang Pasang": "Gelombang Pasang",
    "Abrasi": "Abrasi Pantai",

    # ======================================================
    # CUACA EKSTREM
    # ======================================================

    "Angin Puting Beliung": "Puting Beliung",
    "Puting Beliung": "Puting Beliung",
    "Angin Kencang": "Angin Kencang",
    "Angin Topan": "Angin Topan",
    "Hujan Es": "Hujan Es",
    "Siklon Tropis": "Siklon Tropis",
    "Suhu Udara Ekstrem": "Suhu Udara Ekstrem",

    # ======================================================
    # KEKERINGAN
    # ======================================================

    "Kekeringan": "Kekeringan",
    "Kekeringan Meteorologis": "Kekeringan Meteorologis",
    "Kekeringan Hidrologis": "Kekeringan Hidrologis",
    "Kekeringan Pertanian": "Kekeringan Pertanian",

    # ======================================================
    # KEBAKARAN HUTAN DAN LAHAN
    # ======================================================

    "Kebakaran Hutan": "Kebakaran Hutan",
    "Kebakaran Lahan": "Kebakaran Lahan",
    "Kebakaran Lahan Gambut": "Kebakaran Lahan Gambut",

    # ======================================================
    # GEMPA BUMI
    # ======================================================

    "Gempa Bumi": "Gempa Bumi",
    "Gempa Tektonik": "Gempa Tektonik",
    "Gempa Vulkanik": "Gempa Vulkanik",
    "Gempa Runtuhan": "Gempa Runtuhan",

    # ======================================================
    # TSUNAMI
    # ======================================================

    "Tsunami": "Tsunami",
    "Tsunami Seismogenik": "Tsunami Seismogenik",
    "Tsunami Nonseismik": "Tsunami Nonseismik",
    "Tsunami Lokal": "Tsunami Lokal",
    "Tsunami Regional": "Tsunami Regional",
    "Tsunami Jarak": "Tsunami Jarak",
    "Tsunami Meteorologi": "Tsunami Meteorologi",
    "Mikrotsunami": "Mikrotsunami",

    # ======================================================
    # ERUPSI GUNUNG API
    # ======================================================

    "Gunungapi": "Erupsi Gunung Api",
    "Letusan Gunung Api": "Erupsi Gunung Api",
    "Erupsi Gunung Api": "Erupsi Gunung Api",
    "Awan Panas Guguran": "Awan Panas Guguran",
    "Awan Panas": "Awan Panas",
    "Banjir Lahar": "Banjir Lahar",
    "Hujan Abu Vulkanik": "Hujan Abu Vulkanik",
    "Gas Vulkanik Beracun": "Gas Vulkanik Beracun"

}

df["Disaster type"] = (
    df["Disaster type"]
    .fillna("")
    .str.strip()
    .replace(replace_map)
)

allowed_disaster = [

    # BANJIR
    "Banjir",
    "Banjir Rob",
    "Banjir Bandang",
    "Banjir dan Tanah Longsor",
    "Banjir Drainase & Selokan",
    "Banjir Waduk",
    "Banjir Genangan",
    "Tanggul Jebol",

    # TANAH LONGSOR
    "Tanah Longsor",
    "Gerakan Tanah",

    # GELOMBANG PASANG DAN ABRASI
    "Gelombang Pasang",
    "Abrasi Pantai",

    # CUACA EKSTREM
    "Puting Beliung",
    "Angin Kencang",
    "Angin Topan",
    "Hujan Es",
    "Siklon Tropis",
    "Suhu Udara Ekstrem",

    # KEKERINGAN
    "Kekeringan",
    "Kekeringan Meteorologis",
    "Kekeringan Hidrologis",
    "Kekeringan Pertanian",

    # KEBAKARAN HUTAN DAN LAHAN
    "Kebakaran Hutan",
    "Kebakaran Lahan",
    "Kebakaran Lahan Gambut",

    # GEMPA BUMI
    "Gempa Bumi",
    "Gempa Tektonik",
    "Gempa Vulkanik",
    "Gempa Runtuhan",

    # TSUNAMI
    "Tsunami",
    "Tsunami Seismogenik",
    "Tsunami Nonseismik",
    "Tsunami Lokal",
    "Tsunami Regional",
    "Tsunami Jarak",
    "Tsunami Meteorologi",
    "Mikrotsunami",

    # ERUPSI GUNUNG API
    "Erupsi Gunung Api",
    "Awan Panas Guguran",
    "Awan Panas",
    "Banjir Lahar",
    "Hujan Abu Vulkanik",
    "Gas Vulkanik Beracun"

]

df = df[df["Disaster type"].isin(allowed_disaster)].copy()

# ==========================================================
# MEMBUAT KOLOM YEAR DARI EVENTDATE
# ==========================================================

df["Eventdate"] = pd.to_datetime(
    df["Eventdate"],
    format="%d/%m/%Y %H.%M.%S",
    errors="coerce"
)

df["Year"] = df["Eventdate"].dt.year

print(df[["Eventdate", "Year"]].head())

MEMBACA DATASET
            Eventdate  Year
1 2020-12-10 13:24:00  2020
2 2020-12-09 15:00:00  2020
3 2020-12-09 14:00:00  2020
4 2020-12-09 09:15:00  2020
5 2020-12-08 16:45:00  2020


In [104]:
# Memilih atribut yang digunakan sebagai feature dan target
# ==========================================================
# ATTRIBUTE SELECTION
# ==========================================================

df = df[
[
    "Year",
    "Dead",
    "Missing",
    "Serious Wound",
    "Minor Injuries",
    "Damage",
    "Level"
]
].copy()

In [105]:
# =====================
# DATA PREPROCESSING
# =====================

num_cols=[
    "Dead",
    "Missing",
    "Serious Wound",
    "Minor Injuries"
]

for col in num_cols:

    df[col]=pd.to_numeric(
        df[col],
        errors="coerce"
    )

In [106]:
#MISSING VALUE
df[num_cols]=df[num_cols].fillna(0)

#NILAI NEGATIF
for col in num_cols:
    df[col]=df[col].clip(lower=0)

In [107]:
# Membersihkan teks
df["Damage"] = (
    df["Damage"]
    .fillna("")
    .apply(clean_damage)
)

In [108]:
# ==========================================================
# FEATURE ENGINEERING DAMAGE
# ==========================================================
# Ekstraksi fitur
damage_feature_df = (
    df["Damage"]
    .apply(extract_damage_features)
    .apply(pd.Series)
)

In [109]:
#MERGE FEATURE
df = pd.concat(
    [
        df.drop(columns=["Damage"]),
        damage_feature_df
    ],
    axis=1
)

print(f"Jumlah Data : {len(df)}")
print(f"Jumlah Kolom : {len(df.columns)}")

Jumlah Data : 20691
Jumlah Kolom : 40


In [110]:
# ==========================================================
# TRAIN TEST SPLIT BERDASARKAN TAHUN
# ==========================================================

train_df = df[df["Year"] <= 2023].copy()

test_df = df[df["Year"] >= 2024].copy()

print("="*70)
print("TRAINING")
print(train_df["Year"].value_counts().sort_index())

print()

print("TESTING")
print(test_df["Year"].value_counts().sort_index())

TRAINING
Year
2020    1366
2021    1622
2022    3894
2023    4442
Name: count, dtype: int64

TESTING
Year
2024    4926
2025    4441
Name: count, dtype: int64


In [111]:
#DATA TRANSFORMATION
# ==========================================================
# LABEL ENCODING - TARGET
# ==========================================================
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

# Fit hanya pada data training
label_encoder.fit(train_df["Level"])

y_train_main = label_encoder.transform(train_df["Level"])
y_test_main = label_encoder.transform(test_df["Level"])

X_train_main = train_df.drop(columns=["Level", "Year"])
X_test_main = test_df.drop(columns=["Level", "Year"])

In [112]:
#IMPORT SMOTE
from imblearn.over_sampling import SMOTE

In [113]:
#SMOTE
def smote_data(X_train,y_train):
    sm = SMOTE(
        random_state=42
    )
    return sm.fit_resample(
        X_train,
        y_train
    )

In [114]:
#MODEL UTAMA SETELAH MELAKUKAN SMOTE
X_train_main_sm,\
y_train_main_sm = smote_data(
    X_train_main,
    y_train_main
)

In [115]:
#DATA MINING
#MODEL DECISION TREE
from sklearn.tree import DecisionTreeClassifier
dt = DecisionTreeClassifier(
    criterion="entropy",
    max_depth=10,
    min_samples_leaf=3,
    class_weight="balanced",
    random_state=42
)

dt.fit(
    X_train_main_sm,
    y_train_main_sm
)

pred_dt = dt.predict(X_test_main)

print()
print("="*70)
print("TRAINING SELESAI")
print("="*70)


TRAINING SELESAI


In [116]:
#DECISION RULES
from sklearn.tree import export_text
rules = export_text(
    dt,
    feature_names=list(X_main.columns)
)

print(rules)

|--- House_RR <= 6.02
|   |--- House_Impact <= 10.00
|   |   |--- Land_Ha <= 0.00
|   |   |   |--- Land_Burn_Ha <= 25.38
|   |   |   |   |--- House_RB <= 0.00
|   |   |   |   |   |--- House_Flood <= 20.19
|   |   |   |   |   |   |--- Minor Injuries <= 0.00
|   |   |   |   |   |   |   |--- School_RB <= 0.00
|   |   |   |   |   |   |   |   |--- Worship_RB <= 0.00
|   |   |   |   |   |   |   |   |   |--- House_RR <= 0.98
|   |   |   |   |   |   |   |   |   |   |--- class: 0
|   |   |   |   |   |   |   |   |   |--- House_RR >  0.98
|   |   |   |   |   |   |   |   |   |   |--- class: 0
|   |   |   |   |   |   |   |   |--- Worship_RB >  0.00
|   |   |   |   |   |   |   |   |   |--- Worship_RB <= 0.99
|   |   |   |   |   |   |   |   |   |   |--- class: 2
|   |   |   |   |   |   |   |   |   |--- Worship_RB >  0.99
|   |   |   |   |   |   |   |   |   |   |--- class: 2
|   |   |   |   |   |   |   |--- School_RB >  0.00
|   |   |   |   |   |   |   |   |--- Dead <= 1.12
|   |   |   |   |   |   |  

In [117]:
#DATA MINING
#MODEL RANDOM FOREST
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=500,
    max_depth=12,
    min_samples_leaf=3,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf.fit(
    X_train_main_sm,
    y_train_main_sm
)

pred_rf = rf.predict(X_test_main)

print()
print("="*70)
print("TRAINING SELESAI")
print("="*70)


TRAINING SELESAI


In [118]:
# ==========================================================
# FUNCTION EVALUATION
# ==========================================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

def evaluate_model(model_name, y_true, y_pred):

    print("\n")
    print("="*70)
    print(f"HASIL EVALUASI {model_name}")
    print("="*70)

    accuracy = accuracy_score(
        y_true,
        y_pred
    )

    precision = precision_score(
        y_true,
        y_pred,
        average="weighted"
    )

    recall = recall_score(
        y_true,
        y_pred,
        average="weighted"
    )

    f1 = f1_score(
        y_true,
        y_pred,
        average="weighted"
    )

    hasil = pd.DataFrame({

        "Metric":[
            "Accuracy",
            "Precision",
            "Recall",
            "F1-Score"
        ],

        "Value":[

            accuracy,
            precision,
            recall,
            f1

        ]

    })

    print("\nMETRIK")
    print(hasil.round(4))
    print("\nCLASSIFICATION REPORT\n")
    print(

        classification_report(

            y_true,
            y_pred,
            target_names=label_encoder.classes_,
            digits=4

        )

    )

    cm = confusion_matrix(
        y_true,
        y_pred
    )

    print("\nCONFUSION MATRIX")
    print(cm)
    return hasil

In [ ]:
hasil_dt = evaluate_model(
    "Decision Tree",
    y_test_main,
    pred_dt
)

hasil_rf = evaluate_model(
    "Random Forest",
    y_test_main,
    pred_rf
)



HASIL EVALUASI Decision Tree

METRIK
      Metric   Value
0   Accuracy  0.9382
1  Precision  0.9367
2     Recall  0.9382
3   F1-Score  0.9348

CLASSIFICATION REPORT

              precision    recall  f1-score   support

      RENDAH     0.9611    0.9815    0.9711      8626
      SEDANG     0.7052    0.4355    0.5385       659
      TINGGI     0.2318    0.4268    0.3004        82

    accuracy                         0.9382      9367
   macro avg     0.6327    0.6146    0.6033      9367
weighted avg     0.9367    0.9382    0.9348      9367


CONFUSION MATRIX
[[8466   87   73]
 [ 329  287   43]
 [  14   33   35]]


HASIL EVALUASI Random Forest

METRIK
      Metric   Value
0   Accuracy  0.9416
1  Precision  0.9402
2     Recall  0.9416
3   F1-Score  0.9402

CLASSIFICATION REPORT

              precision    recall  f1-score   support

      RENDAH     0.9664    0.9769    0.9716      8626
      SEDANG     0.6774    0.5417    0.6020       659
      TINGGI     0.3000    0.4390    0.3564    

In [120]:
# ==========================================================
# CROSS VALIDATION
# ==========================================================
cv = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=42
)

pipe_dt = Pipeline([
    ('smote', SMOTE(random_state=42)),
    ('dt', DecisionTreeClassifier(
        criterion="entropy",
        max_depth=10,
        min_samples_leaf=3,
        class_weight="balanced",
        random_state=42
    ))
])

pipe_rf = Pipeline([
    ('smote', SMOTE(random_state=42)),
    ('rf', RandomForestClassifier(
        n_estimators=500,
        max_depth=12,
        min_samples_leaf=3,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

cv_dt = cross_val_score(
    pipe_dt,
    X_main,
    y,
    cv=cv,
    scoring="f1_weighted",
    n_jobs=-1
)

cv_rf = cross_val_score(
    pipe_rf,
    X_main,
    y,
    cv=cv,
    scoring="f1_weighted",
    n_jobs=-1
)
# ==========================================================
# HASIL CROSS VALIDATION
# ==========================================================

cv_result = pd.DataFrame({
    "Model": ["Decision Tree","Random Forest"],
    "Mean F1-Weighted":[
        cv_dt.mean(),
        cv_rf.mean()
    ],
    "Std Dev":[
        cv_dt.std(),
        cv_rf.std()
    ]
})

print("\n")
print("="*70)
print("HASIL CROSS VALIDATION")
print("="*70)

print(cv_result.round(4))
print("\nDecision Tree")
print(np.round(cv_dt,4))
print("\nRandom Forest")
print(np.round(cv_rf,4))



HASIL CROSS VALIDATION
           Model  Mean F1-Weighted  Std Dev
0  Decision Tree            0.9301   0.0031
1  Random Forest            0.9258   0.0053

Decision Tree
[0.9323 0.9288 0.9314 0.9314 0.9311 0.9278 0.9296 0.9246 0.9272 0.9365]

Random Forest
[0.9371 0.9267 0.9212 0.9185 0.9268 0.9217 0.9296 0.9256 0.9203 0.9302]


In [124]:
# ==========================================================
# FEATURE IMPORTANCE
# ==========================================================

importance = pd.DataFrame({
    "Feature": X_train_main.columns,
    "Importance": rf.feature_importances_
})

importance = importance.sort_values(
    by="Importance",
    ascending=False
)

print("\n")
print("="*70)
print("FEATURE IMPORTANCE")
print("="*70)

print(importance)



FEATURE IMPORTANCE
            Feature  Importance
6          House_RR    0.221858
8      House_Impact    0.161717
4          House_RB    0.113282
30          Land_Ha    0.105537
7       House_Flood    0.085158
5          House_RS    0.064377
3    Minor Injuries    0.061391
31     Land_Burn_Ha    0.031596
32      Road_Length    0.025598
0              Dead    0.024245
15       Worship_RB    0.019980
11        School_RR    0.016728
9         School_RB    0.014053
26      Business_RR    0.010978
2     Serious Wound    0.006793
27        Bridge_RB    0.005640
1           Missing    0.005114
37  Drainage_Damage    0.004981
36      Dike_Length    0.004894
24      Business_RB    0.004658
17       Worship_RR    0.003062
25      Business_RS    0.002481
20        Office_RR    0.001797
10        School_RS    0.001598
33          Dike_RB    0.001068
14        Health_RR    0.000438
12        Health_RB    0.000395
21        Market_RB    0.000189
19        Office_RS    0.000172
28        Bridge_RS

In [122]:
# ==========================================================
# PERBANDINGAN MODEL
# ==========================================================

comparison = pd.DataFrame({
    "Model": ["Decision Tree", "Random Forest"],
    "Accuracy": [
        accuracy_score(y_test_main, pred_dt),
        accuracy_score(y_test_main, pred_rf)
    ],
    "Precision": [
        precision_score(y_test_main, pred_dt, average="weighted"),
        precision_score(y_test_main, pred_rf, average="weighted")
    ],
    "Recall": [
        recall_score(y_test_main, pred_dt, average="weighted"),
        recall_score(y_test_main, pred_rf, average="weighted")
    ],
    "F1-Score": [
        f1_score(y_test_main, pred_dt, average="weighted"),
        f1_score(y_test_main, pred_rf, average="weighted")
    ]
})

print("\n")
print("="*70)
print("PERBANDINGAN MODEL")
print("="*70)

print(comparison.round(4))



PERBANDINGAN MODEL
           Model  Accuracy  Precision  Recall  F1-Score
0  Decision Tree    0.9382     0.9367  0.9382    0.9348
1  Random Forest    0.9416     0.9402  0.9416    0.9402


In [125]:
print("="*70)
print("MENYIMPAN MODEL")
print("="*70)

os.makedirs(
    "models",
    exist_ok=True
)

os.makedirs(
    "output",
    exist_ok=True
)

importance.to_csv(
    "output/feature_importance.csv",
    index=False
)

comparison.to_csv(
    "output/model_comparison.csv",
    index=False
)

joblib.dump(
    dt,
    "models/decision_tree.joblib"
)

joblib.dump(
    rf,
    "models/random_forest.joblib"
)

joblib.dump(
    label_encoder,
    "models/label_encoder.joblib"
)

joblib.dump(
    list(X_train_main.columns),
    "models/feature_columns.joblib"
)

joblib.dump(
    mapping,
    "models/label_mapping.joblib"
)

print()
print("="*70)
print("MODEL BERHASIL DISIMPAN")
print("="*70)

print("Decision Tree   : models/decision_tree.joblib")
print("Random Forest   : models/random_forest.joblib")
print("Label Encoder   : models/label_encoder.joblib")
print("Feature Columns : models/feature_columns.joblib")
print("Label Mapping   : models/label_mapping.joblib")

MENYIMPAN MODEL

MODEL BERHASIL DISIMPAN
Decision Tree   : models/decision_tree.joblib
Random Forest   : models/random_forest.joblib
Label Encoder   : models/label_encoder.joblib
Feature Columns : models/feature_columns.joblib
Label Mapping   : models/label_mapping.joblib
